# Playwright with JavaScript — Day 2
## 6 November 2026: Web Element Interactions




## 1. Learning objectives

By the end of this class, students should be able to:

1. Click, double-click and right-click web elements.
2. Enter, replace and clear textbox values.
3. Understand the difference between `fill()`, `type()` and `pressSequentially()`.
4. Work with checkboxes and radio buttons.
5. Use keyboard and mouse actions.
6. Verify interactions using assertions.
7. Write a complete registration-form automation test.


## 3. Click interactions 

Playwright provides different ways to interact with clickable elements.

### Common click methods

| Method | Purpose |
|---|---|
| `click()` | Normal left click |
| `dblclick()` | Double-click |
| `click({ button: 'right' })` | Right-click |
| `click({ button: 'middle' })` | Middle-click |

### Examples

```javascript
// Normal click
await page.getByRole('button', { name: 'Submit' }).click();

// Double-click
await page.locator('#button').dblclick();

// Right-click
await page.locator('#button').click({ button: 'right' });

// Middle-click
await page.locator('#link').click({ button: 'middle' });
```

### Useful click options

```javascript
// Click at a position inside the element
await page.locator('#button').click({
    position: { x: 10, y: 10 }
});

// Force a click (use only when there is a justified reason)
await page.locator('#button').click({
    force: true
});
```

**notes:**
- Playwright normally waits for an element to be actionable before clicking.
- A normal click requires the element to be uniquely identified, visible, stable, enabled and able to receive events.
- Do not use `force: true` as a routine fix for a failing locator. Investigate overlays, visibility and locator accuracy first.


## 4. Textbox interactions  
Textboxes are among the most frequently automated web elements.

### Important methods

| Method | Purpose |
|---|---|
| `fill()` | Replace the current value with text |
| `clear()` | Clear the current value |
| `press()` | Press a keyboard key |
| `pressSequentially()` | Send text character by character |
| `focus()` | Focus the element |
| `inputValue()` | Read the current input value |

### A. Enter text using `fill()`

```javascript
await page.locator('#name').fill('Arun');
```

`fill()` replaces any existing text with the supplied value.

### B. Clear textbox

```javascript
await page.locator('#name').clear();
```

### C. Press a key

```javascript
await page.locator('#name').press('Enter');
await page.locator('#name').press('Tab');
await page.locator('#name').press('Control+A');
```

### D. Type character by character

```javascript
await page.locator('#name').pressSequentially('Arun');
```

This is useful when the application responds to individual keyboard input events.

### E. Read the input value

```javascript
const enteredName = await page.locator('#name').inputValue();

console.log(enteredName);
```

### `fill()` vs `pressSequentially()` vs `press()`

| Method | What it does | Example |
|---|---|---|
| `fill()` | Sets/replaces input value | `fill('Arun')` |
| `pressSequentially()` | Types text character by character | `pressSequentially('Arun')` |
| `press()` | Presses a key or key combination | `press('Enter')` |

**Teaching note:** Prefer `fill()` for normal form entry. Use `pressSequentially()` when the application specifically depends on individual key events.


## 5. Checkboxes and radio buttons (40–55 minutes)

### A. Checkbox

A checkbox allows the user to select or deselect an option. Multiple checkboxes can usually be selected.

HTML example:

```html
<input type="checkbox" id="monday">
<label for="monday">Monday</label>
```

Select a checkbox:

```javascript
await page.locator('#monday').check();
```

Deselect a checkbox:

```javascript
await page.locator('#monday').uncheck();
```

Verify it is selected:

```javascript
await expect(page.locator('#monday')).toBeChecked();
```

Check its current state:

```javascript
const selected = await page.locator('#monday').isChecked();
console.log(selected);
```

### B. Radio button

Radio buttons are generally used when the user should select one option from a group.

HTML example:

```html
<input type="radio" id="male" name="gender" value="male">
<label for="male">Male</label>

<input type="radio" id="female" name="gender" value="female">
<label for="female">Female</label>
```

Select Male:

```javascript
await page.locator('#male').check();
```

Select Female:

```javascript
await page.locator('#female').check();
```

Verify selection:

```javascript
await expect(page.locator('#female')).toBeChecked();
await expect(page.locator('#male')).not.toBeChecked();
```

### Checkbox vs radio button

| Checkbox | Radio button |
|---|---|
| Allows multiple selections in a group | Usually one selection per group |
| Can be checked or unchecked | Selecting another option in the same group deselects the previous one |
| Example: Hobbies | Example: Gender selection |

**Important methods:** `check()`, `uncheck()`, `isChecked()` and `toBeChecked()`.


## 6. Keyboard and mouse interactions (55–70 minutes)

### A. Keyboard actions

Playwright provides `page.keyboard` for keyboard operations.

```javascript
// Press a key
await page.keyboard.press('Enter');

// Select all text
await page.keyboard.press('Control+A');

// Copy
await page.keyboard.press('Control+C');

// Paste
await page.keyboard.press('Control+V');

// Type text
await page.keyboard.type('Playwright');
```

You can also use locator-level keyboard actions:

```javascript
await page.locator('#name').press('Enter');
await page.locator('#name').press('Control+A');
```

### B. Mouse actions

Playwright provides `page.mouse` for low-level mouse interactions.

```javascript
// Move mouse to coordinates
await page.mouse.move(200, 300);

// Click at coordinates
await page.mouse.click(200, 300);

// Double-click at coordinates
await page.mouse.dblclick(200, 300);

// Mouse down and mouse up
await page.mouse.down();
await page.mouse.up();
```

### C. Hover over an element

For normal element hover, prefer the locator API:

```javascript
await page.locator('#menu').hover();
```

This is useful for menus that appear when the mouse moves over an element.

### D. Drag and drop — introduction

```javascript
await page.locator('#source').dragTo(
    page.locator('#target')
);
```

Use `dragTo()` when the page supports standard drag-and-drop interactions. Some custom drag interfaces may require mouse movement and button actions.

**Teaching notes:**
- Locator actions are usually more robust than coordinate-based mouse actions.
- Use `page.mouse` when coordinates or low-level pointer control are specifically required.
- Keyboard shortcuts can differ between operating systems. For example, Windows/Linux commonly use `Control`, while macOS commonly uses `Meta`.


## 7. Complete practical — Registration form (70–85 minutes)

**Practice website:** https://testautomationpractice.blogspot.com/

The form includes fields such as Name, Email, Phone, Gender and Days.

Save the following code as `tests/day2-interactions.spec.js`:

```javascript
const { test, expect } = require('@playwright/test');

test('Handle registration form elements', async ({ page }) => {

    // Step 1: Open practice website
    await page.goto('https://testautomationpractice.blogspot.com/');

    // Step 2: Enter name
    const name = page.locator('#name');
    await name.fill('Arun');
    await expect(name).toHaveValue('Arun');

    // Step 3: Enter email
    const email = page.locator('#email');
    await email.fill('arun@example.com');
    await expect(email).toHaveValue('arun@example.com');

    // Step 4: Enter phone number
    const phone = page.locator('#phone');
    await phone.fill('9876543210');
    await expect(phone).toHaveValue('9876543210');

    // Step 5: Select gender
    await page.locator('#male').check();
    await expect(page.locator('#male')).toBeChecked();

    // Step 6: Select multiple days
    await page.locator('#monday').check();
    await page.locator('#wednesday').check();
    await page.locator('#friday').check();

    // Step 7: Verify selected days
    await expect(page.locator('#monday')).toBeChecked();
    await expect(page.locator('#wednesday')).toBeChecked();
    await expect(page.locator('#friday')).toBeChecked();

    // Step 8: Verify an unselected day
    await expect(page.locator('#sunday')).not.toBeChecked();

    // Step 9: Clear the name field and verify
    await name.clear();
    await expect(name).toHaveValue('');

});
```

### Explain the test step by step

1. `page.goto()` opens the practice website.
2. `locator('#name')` identifies the Name textbox.
3. `fill()` enters the name.
4. `toHaveValue()` verifies the entered value.
5. The same approach is used for Email and Phone.
6. `check()` selects the Male radio button.
7. `check()` selects multiple day checkboxes.
8. `toBeChecked()` verifies the selected options.
9. `.not.toBeChecked()` verifies that Sunday is not selected.
10. `clear()` clears the Name textbox.

**Note:** This example assumes the practice page exposes the listed IDs. Inspect the live page if a selector does not match; do not assume a locator is correct without verifying it.


## 8. Five-minute recap (85–90 minutes)

Ask students:

1. What is the difference between `click()` and `dblclick()`?
2. What is the difference between `fill()` and `pressSequentially()`?
3. How do we select a checkbox?
4. How do we verify whether a checkbox is selected?
5. What is the difference between `page.keyboard` and `page.mouse`?
6. When should we use `force: true`?

### Quick reference

| Requirement | Method |
|---|---|
| Normal click | `click()` |
| Double-click | `dblclick()` |
| Right-click | `click({ button: 'right' })` |
| Enter/replace text | `fill()` |
| Clear text | `clear()` |
| Type character by character | `pressSequentially()` |
| Press Enter | `press('Enter')` |
| Select checkbox/radio | `check()` |
| Deselect checkbox | `uncheck()` |
| Check state immediately | `isChecked()` |
| Assert selected state | `toBeChecked()` |
| Hover | `hover()` |
| Drag and drop | `dragTo()` |
| Keyboard action | `page.keyboard.press()` |
| Coordinate mouse action | `page.mouse.click()` |


## 9. Student practical assignment (additional 30 minutes)

Use https://testautomationpractice.blogspot.com/

Complete these tasks independently:

1. Enter your full name and verify it.
2. Enter an email address and verify it.
3. Enter a phone number and verify it.
4. Clear the phone textbox and verify it is empty.
5. Select the Female radio button.
6. Verify Female is selected and Male is not selected.
7. Select at least three days using checkboxes.
8. Verify all selected days using assertions.
9. Deselect one selected day and verify it is unchecked.
10. Hover over a menu or another suitable element.
11. Use a keyboard action to move between fields or submit a form.
12. Count the selected checkboxes where practical.

### Assignment rules

- Write your own locators.
- Use assertions after important interactions.
- Do not use `force: true` to hide locator or interaction problems.
- Run the test in headed mode and observe each action.
- If a locator fails, inspect the actual webpage and correct the locator.


## 10. Day 2 completion checklist

- [ ] Perform a normal click.
- [ ] Perform a double-click.
- [ ] Perform a right-click.
- [ ] Enter and replace textbox values using `fill()`.
- [ ] Clear a textbox using `clear()`.
- [ ] Use `press()` and `pressSequentially()`.
- [ ] Select and deselect checkboxes.
- [ ] Select radio buttons.
- [ ] Verify checked and unchecked states.
- [ ] Perform keyboard interactions.
- [ ] Perform mouse interactions.
- [ ] Use hover and drag-and-drop where applicable.
- [ ] Complete the registration-form test.
- [ ] Complete the independent assignment.


## Teacher's notes

1. Begin with a quick revision of Day 1 locators and assertions.
2. Demonstrate each action in a visible browser so students can observe it.
3. Explain the difference between an action and an assertion.
4. For every interaction, ask students how they would verify the result.
5. Explain that `check()` and `uncheck()` are safer than clicking a checkbox blindly when the desired state is known.
6. Prefer locator-based actions over coordinate-based mouse actions.
7. Do not encourage `force: true` as a default workaround.
8. Verify selectors against the live website before class. Practice websites can change their DOM.

**Official references:**
- Locator actions: https://playwright.dev/docs/input
- Locators: https://playwright.dev/docs/locators
- Assertions: https://playwright.dev/docs/test-assertions

---

**End of Day 2 — 6 November 2026**
